<a href="https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Query 1 — Grain check

In [39]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [40]:
rel = "hf://datasets/FlyRank/internship-warehouse"

# dim_clients is a single flat file
total_clients = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{rel}/dim_clients.parquet')
""").df()
print("dim_clients rows:", total_clients)

# March 2026 = our chosen mid-panel month
march_count = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
print("March 2026 daily fact rows:", march_count)

dim_clients rows:      n
0  104
March 2026 daily fact rows:          n
0  9841378


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

##  **Unit of analysis + time window**

**Unit of analysis:** one row = one content item, on one report date, for one client
(a "page-day" — content_hash_id × client_hash_id × report_date), from
fact_content_daily_performance.

**Time window for this contract:** March 2026 (month=2026-03), a mid-panel month — not
the final month, since the last month is a natural label-outcome window and shouldn't be
used to develop label logic (per the card's warning).

**Why page-day, not page:** Lane 2's decision (which page to review first) is ultimately
about one page, but the daily fact table's grain is finer — one row per page per day.
I'll aggregate up to one-row-per-page for the actual feature frame in Section 3, but the
raw grain I'm querying here is page-day, and I verify that below.

In [41]:
month = "2026-03"

grain_check = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id, report_date,
        COUNT(*) AS c
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()

print("Rows violating page-day grain (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating page-day grain (should be empty):
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, c]
Index: []


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [42]:
schema = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')
    LIMIT 1
""").df()
print(schema.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


##  **Fields: feature / label / context / excluded**

**Context (grouping, joining, filtering — never features):**
- `content_hash_id`, `client_hash_id` — pseudonymous IDs, used only for grouping/joining/splitting.
- `report_date`, `month` — used to define time windows, not model inputs directly (may derive
  age/recency features from them later, but the raw date itself isn't a feature).
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — availability
  flags. These decide WHICH rows are safe to use for GSC- or GA4-based features; they are
  filters, not predictive signals themselves.

**Feature (knowable before the decision moment, safe to use):**
- `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position` — observed search
  performance, known as of the report date.
- `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`
  — observed on-site engagement, known as of the report date.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`,
  `sessions_ai` — traffic-source breakdown, observed.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` —
  AI-platform referral breakdown, observed (though sparse — per the data skill, only ~30k of
  78.8M daily rows have any AI sessions at all).
- `scroll_events` — observed engagement depth signal.

**Label / proxy:**
- None of these raw columns is a label by itself. This table has no `trend_direction`-style
  column like the starter CSV does. A label must be DERIVED — comparing a page's signals in an
  earlier window against a later window (e.g. prior 90 days vs. next 30 days), never read
  directly off one row. This is exactly the "future observed outcome" the lane guide
  recommends over the starter's current-window proxy.

**Excluded:**
- Any raw client name, domain, URL, or query text — never shipped in this release; nothing to
  exclude because it was never present (already scrambled out).
- No FlyRank product-decision flags (health_score, priority_score, action_type) exist in this
  table either — same reason, they were deliberately never included upstream.
- **Deliberately excluded despite being available:** the individual AI-platform columns
  (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other). They
  exist in the schema, but per the data skill, only ~30,177 of 78.8M daily rows across the
  whole warehouse have any AI session at all — far too sparse to build a reliable Lane 2
  feature from without risking a model that "looks impressive and means nothing" on such
  rare positives.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## **Verify it with queries**

Three verification queries below: grain (done in Section 1), counts + date span, and
availability filtered with IS TRUE.

## Query 2 — Row count + date span for March 2026

In [43]:
counts_and_span = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')
""").df()
print(counts_and_span)

   total_rows  n_clients  n_content_items   min_date   max_date
0     9841378         55           331437 2026-03-01 2026-03-31


## Query 3 — Availability, filtered with IS TRUE

In [44]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')
""").df()
print(availability)
print()
print(f"Share with GSC data: {availability['gsc_available_rows'][0] / availability['total_rows'][0]:.1%}")
print(f"Share with GA4 data: {availability['ga4_available_rows'][0] / availability['total_rows'][0]:.1%}")

   total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
0     9841378           3611061.0            413966.0             364347.0

Share with GSC data: 36.7%
Share with GA4 data: 4.2%


These numbers match what the lane guide's density table predicted: GSC availability is much
higher than GA4 in this snapshot, since not every client has GA4 connected or a GA4 start
date this early. Any GA4-based feature must filter on `ga4_data_available IS TRUE`, or zeros
from unavailable rows will masquerade as "zero engagement."

### 5 features for Lane 2, built from March 2026

In [45]:
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        COUNT(DISTINCT report_date) AS days_with_any_row,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(CASE WHEN gsc_data_available IS TRUE THEN gsc_avg_position END) AS avg_position_month,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_month,
        MAX(report_date) - MIN(report_date) AS days_span_seen
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id
    ORDER BY content_hash_id
""").df()

print(features_df.shape)
features_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 8)


,content_hash_id,client_hash_id,days_with_any_row,impressions_month,clicks_month,avg_position_month,sessions_month,days_span_seen
0,content_000005d4ced12088,client_9958f0a7ae1df715,31,86.0,0.0,72.854861,0.0,30
1,content_00001e488b74b799,client_625b6439094e23e4,31,0.0,0.0,NaN,0.0,30
2,content_00007bd2985b77c3,client_73cda7b4e4f265ea,31,47.0,0.0,5.269565,0.0,30
3,content_00008950670cb6b5,client_def0955f7a377868,31,0.0,0.0,NaN,2.0,30
4,content_0000a348850eb1fc,client_3ffa76342f366962,29,0.0,0.0,NaN,1.0,28
5,content_0000c57e204651e5,client_3ffa76342f366962,29,0.0,0.0,NaN,0.0,28
6,content_0000cd28fbda69f3,client_3ffa76342f366962,30,29.0,0.0,4.251282,0.0,29
7,content_0000d31f3926ea12,client_3ffa76342f366962,29,0.0,0.0,NaN,0.0,28
8,content_0000d495bfbfb4a8,client_2094c6eb080311d5,31,15.0,0.0,3.333333,1.0,30
9,content_0000feb69f0f60db,client_2b4306c3ed003f01,31,0.0,0.0,NaN,0.0,30


**Reproducibility note:** the feature-building query used GROUP BY without ORDER BY,
so row order (and therefore the train/test split) varied slightly between reruns, causing
small score drift (~0.63–0.64, ~0.99). Fixed by adding ORDER BY content_hash_id for a
stable, reproducible split.

**The five features, and why each is knowable at the decision moment:**

1. **impressions_month** — knowable because it's a sum of past search impressions already
   observed by the end of March; nothing from the future is used.
2. **clicks_month** — same reasoning: a sum of past observed clicks, known as of month-end.
3. **avg_position_month** — an average of past observed search rankings, filtered to only
   rows where `gsc_data_available IS TRUE`, so unavailable rows don't silently pull it toward
   zero.
4. **sessions_month** — past observed GA4 sessions, filtered on `ga4_data_available IS TRUE`
   for the same reason; zero-filled unavailable rows would otherwise look like "zero
   engagement" instead of "no data yet."
5. **days_with_any_row** — a count of how many days in March this page had any tracked
   activity at all; knowable as a simple count of past rows, useful as a coverage/reliability
   signal (a page seen on 2 days out of 31 deserves less trust than one seen on all 31).

None of these five uses any information from April 2026 or later — every one is computable
using only rows dated on or before March 31, 2026.

In [46]:
# Build a simple proxy label for this exercise: "underperforming" page
# (low visibility AND weak clicks) - not a perfect label, just something to demo leakage on.
features_df["is_underperforming"] = (
    (features_df["avg_position_month"] > 20) &
    (features_df["clicks_month"] < 5)
).astype(int)

print(features_df["is_underperforming"].value_counts())
print("Base rate:", features_df["is_underperforming"].mean().round(3))

is_underperforming
0    289833
1     41604
Name: count, dtype: int64
Base rate: 0.126


In [47]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

honest_features = ["impressions_month", "clicks_month", "avg_position_month",
                    "sessions_month", "days_with_any_row"]

X = features_df[honest_features].fillna(0)
y = features_df["is_underperforming"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
honest_tree.fit(X_train, y_train)

honest_score = honest_tree.score(X_test, y_test)
print(f"Honest accuracy (5 real features only): {honest_score:.3f}")
print()
print(export_text(honest_tree, feature_names=honest_features))

Honest accuracy (5 real features only): 1.000

|--- avg_position_month <= 20.00
|   |--- class: 0
|--- avg_position_month >  20.00
|   |--- clicks_month <= 4.50
|   |   |--- class: 1
|   |--- clicks_month >  4.50
|   |   |--- class: 0



In [48]:
honest_features_fixed = ["impressions_month", "sessions_month", "days_with_any_row"]

X = features_df[honest_features_fixed].fillna(0)
y = features_df["is_underperforming"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
honest_tree.fit(X_train, y_train)

honest_score = honest_tree.score(X_test, y_test)
print(f"Honest accuracy (features NOT used to define the label): {honest_score:.3f}")
print()
print(export_text(honest_tree, feature_names=honest_features_fixed))

Honest accuracy (features NOT used to define the label): 0.643

|--- impressions_month <= 0.50
|   |--- class: 0
|--- impressions_month >  0.50
|   |--- impressions_month <= 781.50
|   |   |--- days_with_any_row <= 30.50
|   |   |   |--- class: 1
|   |   |--- days_with_any_row >  30.50
|   |   |   |--- class: 1
|   |--- impressions_month >  781.50
|   |   |--- impressions_month <= 3631.50
|   |   |   |--- class: 1
|   |   |--- impressions_month >  3631.50
|   |   |   |--- class: 0



**Note on the first attempt:** my first "honest" feature set accidentally included
avg_position_month and clicks_month — the exact two fields used to DEFINE the label itself.
That produced a suspicious 1.000 accuracy, which was really just the tree re-deriving its own
label rule, not learning anything. I fixed this by excluding those two fields from the honest
baseline, keeping only features genuinely independent of the label definition.

In [49]:
# DELIBERATE LEAK: this column is directly derived from the same rule used to build the
# label. A model should never see this - it's the answer in disguise.
features_df["leaky_position_flag"] = (features_df["avg_position_month"] > 20).astype(int)

leaky_features = honest_features_fixed + ["leaky_position_flag"]

X_leak = features_df[leaky_features].fillna(0)
y = features_df["is_underperforming"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.3, random_state=42, stratify=y
)

leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
leaky_tree.fit(X_train, y_train)

leaky_score = leaky_tree.score(X_test, y_test)
print(f"Honest accuracy (no leak):        {honest_score:.3f}")
print(f"'Leaky' accuracy (with the trap): {leaky_score:.3f}  <- looks amazing, but it's cheating")
print()
print(export_text(leaky_tree, feature_names=leaky_features))

Honest accuracy (no leak):        0.643
'Leaky' accuracy (with the trap): 0.990  <- looks amazing, but it's cheating

|--- leaky_position_flag <= 0.50
|   |--- class: 0
|--- leaky_position_flag >  0.50
|   |--- impressions_month <= 3767.50
|   |   |--- impressions_month <= 1471.00
|   |   |   |--- class: 1
|   |   |--- impressions_month >  1471.00
|   |   |   |--- class: 1
|   |--- impressions_month >  3767.50
|   |   |--- impressions_month <= 12749.00
|   |   |   |--- class: 1
|   |   |--- impressions_month >  12749.00
|   |   |   |--- class: 1



### The trap, sprung on purpose

I added one deliberate leak: `leaky_position_flag = (avg_position_month > 20)`. This is
directly derived from `avg_position_month`, one of the two fields that DEFINE the label
`is_underperforming` itself. It should never be a real feature — it's the answer, disguised
as an input.

**Honest accuracy (no leak): 0.643**
**"Leaky" accuracy (with the trap): 0.990** — a jump of nearly 35 points.

The printed tree confirms exactly why: the very first, dominant split is on
`leaky_position_flag` — the model didn't learn a real pattern, it just found the shortcut
back to the label's own definition and used it. This is precisely the same failure mode as
`trend_pct` in Notebook 02 (starter data): a feature that looks powerful only because it
secretly encodes the target.

**I removed `leaky_position_flag` afterward and kept 0.643 as the honest number to report
going forward** — the real, defensible signal in this data, not the inflated one.

In [50]:
# Remove the deliberate leak - keep only the honest result going forward.
features_df = features_df.drop(columns=["leaky_position_flag"])
print("Leaky column removed. Final honest accuracy to report:", round(honest_score, 3))

Leaky column removed. Final honest accuracy to report: 0.643


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## **Data limits**

**GA4 coverage is thin in this slice.** Only 4.2% of March 2026 rows have `ga4_data_available
= TRUE` (413,966 of 9,841,378), and only 3.7% have both GSC and GA4 together. This means any
engagement-based feature (sessions, pageviews, engagement rate) will only be computable for a
small fraction of pages in a given month — most of my Lane 2 features will need to lean on GSC
signals (impressions, clicks, position) as the primary source, with GA4 as a secondary,
sparser layer.

**Only 55 clients appear in March 2026,** compared to 104 total in dim_clients — meaning
roughly half of all clients have no daily fact rows in this particular month. Combined with
the lane guide's warning about unbalanced client history (`gsc_data_start` /
`ga4_data_start` differ per client), this means I can't assume a consistent client roster
across months — any cross-month comparison needs to explicitly handle clients entering or
leaving the panel.

**This data cannot prove causation.** Even with a properly future-windowed label (prior 90
days → next 30 days), the strongest honest claim is "pages with these signals were more
likely to decline/recover" — never "refreshing this page will cause recovery," which would
require a controlled experiment this dataset doesn't provide.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.